<a href="https://colab.research.google.com/github/2001lida/PythonLession2/blob/hw_7/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B57.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin

pd.set_option('display.max_colwidth', 300)


In [2]:
BASE_URL = 'https://lifehacker.ru'
CATEGORY_URL = 'https://lifehacker.ru/topics/technology/'
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/131.0.0.0 Safari/537.36'
    )
}
SLEEP_SEC = 0.3

for i in range(1, 4):
    print(CATEGORY_URL if i == 1 else f'{CATEGORY_URL}?page={i}')


https://lifehacker.ru/topics/technology/
https://lifehacker.ru/topics/technology/?page=2
https://lifehacker.ru/topics/technology/?page=3


In [3]:
def get_soup(url: str) -> BeautifulSoup:
    response = requests.get(url, headers=HEADERS, timeout=20)
    response.raise_for_status()
    return BeautifulSoup(response.text, 'lxml')


def is_article_url(href: str) -> bool:
    if not href:
        return False

    full_url = urljoin(BASE_URL, href)

    if not full_url.startswith(BASE_URL):
        return False

    banned_parts = [
        '/topics/',
        '/tag/',
        '/category/',
        '/page/',
        '/wp-content/',
        '/wp-json/',
        '#comments',
        '/comments',
        '/amp/',
        '/specials/',
        '/podcasts/',
        '/courses/',
    ]
    if any(part in full_url for part in banned_parts):
        return False

    path = full_url.replace(BASE_URL, '')
    return bool(re.fullmatch(r'/[^/]+/?', path))


def collect_article_links(pages: int = 10) -> list[str]:
    article_links = []

    for page_num in tqdm(range(1, pages + 1), desc='Собираем ссылки со страниц рубрики'):
        url = CATEGORY_URL if page_num == 1 else f'{CATEGORY_URL}?page={page_num}'
        soup = get_soup(url)

        for a_tag in soup.find_all('a', href=True):
            href = a_tag.get('href')
            if is_article_url(href):
                article_links.append(urljoin(BASE_URL, href))

        time.sleep(SLEEP_SEC)

    unique_links = list(dict.fromkeys(article_links))
    return unique_links


def extract_title(soup: BeautifulSoup) -> str | None:
    h1 = soup.find('h1')
    if h1:
        return h1.get_text(' ', strip=True)

    if soup.title:
        return soup.title.get_text(' ', strip=True)

    return None


def extract_text(soup: BeautifulSoup) -> str | None:
    candidates = [
        soup.find('article'),
        soup.find('div', attrs={'itemprop': 'articleBody'}),
        soup.find('div', class_=re.compile(r'article|content|post', re.I)),
        soup.find('main'),
    ]

    for block in candidates:
        if block is None:
            continue

        paragraphs = block.find_all(['p', 'li'])
        text = '\n'.join(
            p.get_text(' ', strip=True)
            for p in paragraphs
            if p.get_text(' ', strip=True)
        ).strip()

        if len(text) > 200:
            return text

    return None


def parse_articles(urls: list[str]) -> list[dict]:
    result = []

    for url in tqdm(urls, desc='Парсим статьи'):
        try:
            soup = get_soup(url)

            title = extract_title(soup)
            text = extract_text(soup)

            if title and text:
                result.append({
                    'url': url,
                    'title': title,
                    'text': text,
                })

            time.sleep(SLEEP_SEC)

        except Exception as e:
            print(f'Ошибка при обработке {url}: {e}')

    return result


In [4]:
parsed_urls = collect_article_links(pages=10)

print(f'Найдено ссылок: {len(parsed_urls)}')
parsed_urls[:10]


Собираем ссылки со страниц рубрики: 100%|██████████| 10/10 [00:30<00:00,  3.06s/it]

Найдено ссылок: 314


['https://lifehacker.ru/recipes/',
 'https://lifehacker.ru/health/',
 'https://lifehacker.ru/reklama/',
 'https://lifehacker.ru/anons-xiaomi-book-pro-14/',
 'https://lifehacker.ru/avtomobilnaya-zaryadka-ugreen-so-skidkoj/',
 'https://lifehacker.ru/besplatnyi-kurs-po-ii-v-uchebe/',
 'https://lifehacker.ru/bezrabotica-sredi-zumerov/',
 'https://lifehacker.ru/internetometr-vyshel-na-ios-i-android/',
 'https://lifehacker.ru/kak-skachat-dannye-iz-telegram/',
 'https://lifehacker.ru/comet-vyshel-na-iphone/']

In [5]:
result = parse_articles(parsed_urls)

data = pd.DataFrame(result)
print(data.shape)
data.head()


Парсим статьи:  12%|█▏        | 39/314 [00:49<05:13,  1.14s/it]/tmp/ipykernel_8818/4190382467.py:4: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  return BeautifulSoup(response.text, 'lxml')
Парсим статьи: 100%|██████████| 314/314 [09:01<00:00,  1.73s/it]

(312, 3)


,url,title,text
0,https://lifehacker.ru/health/,"Лайфхакер Здоровье: рассказываем, как принимать решения о здоровье, опираясь на принципы доказательной медицины","Принимайте решения о своём здоровье на основе достоверной информации\nМы проверили все факты и отвечаем за свои слова!\n100+\nврачей-рецензентов\n5 000+\nстатей о здоровье\nО разделе\nкак мы работаем\nБолезни\nПрепараты\nПервая помощь\nАнализы\nОбещание\nМы создали медицинский раздел, чтобы вы н..."
1,https://lifehacker.ru/reklama/,Реклама,"Лайфхакер читает более 20 миллионов человек в месяц. Мы придумываем идеи индивидуально под каждого клиента, чтобы выполнить именно его задачи. У нас есть много крутых форматов и нет шаблонных решений.\nМедиакит Лайфхакера\nПрайс\nЗа последние два года мы выпустили больше 1 000 партнёрских матери..."
2,https://lifehacker.ru/anons-xiaomi-book-pro-14/,Xiaomi представила тонкий и лёгкий премиум-ноутбук Book Pro 14,"Xiaomi официально вернулась на рынок ноутбуков, представив Book Pro 14 на Windows 11. Толщина устройства составляет всего 14,95 миллиметра, а вес — 1,08 килограмма. Новинка позиционируется как премиальная модель.\nКорпус выполнен из магниевого сплава, а крышка — из углеродного волокна. В компани..."
3,https://lifehacker.ru/avtomobilnaya-zaryadka-ugreen-so-skidkoj/,Надо брать: мощная автомобильная зарядка от Ugreen со скидкой 48%,"На AliExpress в самом разгаре « Великая китайская распродажа », а мы продолжаем делиться товарами с хорошими скидками от проверенных брендов. Сегодня хотим предложить мощное автомобильное зарядное устройство от Ugreen, которое справится с подпиткой всех ваших гаджетов.\nУ адаптера три порта: два..."
4,https://lifehacker.ru/besplatnyi-kurs-po-ii-v-uchebe/,«Яндекс» научит писать курсовые и дипломы с помощью ИИ,"«Яндекс» расширяет проект по применению ИИ в дипломных работах, который он ведёт совместно с факультетом компьютерных наук НИУ ВШЭ. В 2026 году к нему подключатся 20 вузов — почти вдвое больше, чем годом ранее. Параллельно компания открыла доступ к бесплатному онлайн-курсу по использованию ИИ в ..."


## 5. Сохранение результата

In [6]:
data.to_csv('lifehacker_technology_10_pages.csv', index=False, encoding='utf-8-sig')
data.to_excel('lifehacker_technology_10_pages.xlsx', index=False)

print('Файлы сохранены:')
print('- lifehacker_technology_10_pages.csv')
print('- lifehacker_technology_10_pages.xlsx')


Файлы сохранены:
- lifehacker_technology_10_pages.csv
- lifehacker_technology_10_pages.xlsx
